# 01 · IHP SG13G2 PDK, ngspice and LUT extraction for novaad

This notebook prepares everything novaad needs to size transistors in the open **IHP SG13G2 130 nm BiCMOS** process:

1. Install the SG13G2 PDK ([IHP-GmbH/IHP-Open-PDK](https://github.com/IHP-GmbH/IHP-Open-PDK)) with [**ciel**](https://github.com/fossi-foundation/ciel).
2. Install **ngspice** and compile the PSP103 MOSFET models to OSDI with [**OpenVAF-Reloaded**](https://github.com/OpenVAF/OpenVAF-Reloaded).
3. Run a SPICE testbench that sweeps a single `sg13_lv_nmos` / `sg13_lv_pmos` (1.2 V core devices, Lmin = 130 nm) and writes the **look-up tables (LUTs)** novaad reads.

Notebooks 02–05 use the LUTs written here.

> **Why not gf180mcu?** Its open PDK has no 1.8 V core devices: the shortest MOSFETs are the 3.3 V `nfet_03v3` / `pfet_03v3` at L = 0.28 µm, and `l=0.18u` fails with *could not find a valid modelname*. Open issue: [gdsfactory/gf180mcu#205](https://github.com/gdsfactory/gf180mcu/issues/205).


## 1. Install the IHP SG13G2 PDK with ciel

`ciel` is the PDK version manager from the FOSSi Foundation (successor of `volare`). It downloads prebuilt releases of the open PDKs and places them under a PDK root (default `~/.ciel`).


In [1]:
# Global PDK settings: edit these
PDK_FAMILY = "ihp-sg13g2"
PDK_VARIANT = "ihp-sg13g2"  # SG13G2 has a single variant
PDK_VERSION = None          # None -> latest release, or pin a commit hash from `ciel ls-remote`
PDK_ROOT = "~/.ciel"


In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PDK_ROOT = Path(PDK_ROOT).expanduser()

# install ciel into this environment if it isn't on PATH yet
if shutil.which("ciel") is None and not (Path(sys.executable).parent / "ciel").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ciel"], check=True)
CIEL = shutil.which("ciel") or str(Path(sys.executable).parent / "ciel")
print(subprocess.run([CIEL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

Ciel v3.2.0 ©2022-2025 Efabless Corporation and Contributors


In [3]:
versions = subprocess.run(
    [CIEL, "ls-remote", "--pdk-family", PDK_FAMILY], capture_output=True, text=True, check=True
).stdout.split()
version = PDK_VERSION or versions[0]
print(f"{len(versions)} {PDK_FAMILY} releases available, enabling {version}")

proc = subprocess.run(
    [CIEL, "enable", "--pdk-root", str(PDK_ROOT), "--pdk-family", PDK_FAMILY, version],
    capture_output=True, text=True, check=True,
)
print(f"enabled {PDK_FAMILY} {version} in {PDK_ROOT}")

MODELS = PDK_ROOT / PDK_VARIANT / "libs.tech" / "ngspice"
assert (MODELS / "models" / "cornerMOSlv.lib").is_file(), f"ngspice models not found in {MODELS}"
sorted(p.name for p in (MODELS / "models").glob("*mos*"))

27 ihp-sg13g2 releases available, enabling cc9afba57e0ec1b5ba77e6bbb638ba9dbe5866b8


enabled ihp-sg13g2 cc9afba57e0ec1b5ba77e6bbb638ba9dbe5866b8 in /Users/gfdasdias/.ciel


['sg13g2_moscap_mismatch.lib',
 'sg13g2_moscap_mod.lib',
 'sg13g2_moscap_mod_mismatch.lib',
 'sg13g2_moscap_parm.lib',
 'sg13g2_moscap_stat.lib',
 'sg13g2_moshv_mismatch.lib',
 'sg13g2_moshv_mod.lib',
 'sg13g2_moshv_mod_mismatch.lib',
 'sg13g2_moshv_parm.lib',
 'sg13g2_moshv_stat.lib',
 'sg13g2_moslv_mismatch.lib',
 'sg13g2_moslv_mod.lib',
 'sg13g2_moslv_mod_mismatch.lib',
 'sg13g2_moslv_parm.lib',
 'sg13g2_moslv_stat.lib']

The low-voltage MOSFETs `sg13_lv_nmos` / `sg13_lv_pmos` (1.2 V, L ≥ 130 nm) live in `cornerMOSlv.lib`, with the corners `mos_tt`, `mos_ss`, `mos_ff`, `mos_sf` and `mos_fs`. Each one is a subcircuit around a single **PSP103** device. PSP103 is a Verilog-A compact model, which ngspice loads as a compiled **OSDI** library. Instances therefore use `X`: `XM1 d g s b sg13_lv_nmos w=... l=... ng=...`, where `w` is the total width, split over `ng` fingers.

## 2. Install ngspice and compile the OSDI models


In [4]:
import platform

if shutil.which("ngspice") is None:
    if platform.system() == "Darwin":
        subprocess.run(["brew", "install", "ngspice"], check=True)
    else:
        raise SystemExit("Install ngspice with your package manager, e.g. `sudo apt install ngspice` (version >= 40, built with OSDI support)")
print(subprocess.run(["ngspice", "-v"], capture_output=True, text=True).stdout)

******
** ngspice-47 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2026, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
******



The PDK ships the PSP103 Verilog-A sources, not binaries for every platform. **OpenVAF-Reloaded** compiles them into `libs.tech/ngspice/osdi`, where `spice.run` loads them. If `openvaf-r` isn't on `PATH`, the release build for this platform is downloaded to `~/.local/opt/openvaf-r`. On macOS, its bundled libraries ship with invalid code signatures, so macOS kills `openvaf-r` at launch. The cell re-signs them ad hoc.


In [5]:
import tarfile
import urllib.request

OPENVAF_VERSION = "v24.0.2mob"                         # OpenVAF-Reloaded release
OPENVAF_HOME = Path("~/.local/opt/openvaf-r").expanduser()
OSDI_MODELS = {"psp103": "psp103", "psp103_nqs": "psp103"}   # model -> Verilog-A source directory


def openvaf() -> str:
    """Path of openvaf-r: the one on PATH, else the release build for this platform (downloaded once)."""
    if exe := shutil.which("openvaf-r"):
        return exe
    exe = OPENVAF_HOME / "bin" / "openvaf-r"
    if not exe.exists():
        os_name = {"Darwin": "macos", "Linux": "linux"}[platform.system()]
        arch = {"arm64": "aarch64", "aarch64": "aarch64", "x86_64": "x86_64"}[platform.machine()]
        name = f"openvaf-r-{OPENVAF_VERSION}-{os_name}-{arch}"
        url = f"https://github.com/OpenVAF/OpenVAF-Reloaded/releases/download/{OPENVAF_VERSION}/{name}.tar.gz"
        archive, _ = urllib.request.urlretrieve(url)
        with tarfile.open(archive) as tar:
            tar.extractall(OPENVAF_HOME.parent, filter="data")
        (OPENVAF_HOME.parent / name).rename(OPENVAF_HOME)
        if os_name == "macos":
            dylibs = [str(p) for p in (OPENVAF_HOME / "lib").glob("*.dylib")]
            subprocess.run(["codesign", "--force", "-s", "-", *dylibs], check=True)
    return str(exe)


osdi_dir = MODELS / "osdi"
osdi_dir.mkdir(exist_ok=True)
for model, src in OSDI_MODELS.items():
    if not (osdi_dir / f"{model}.osdi").exists():
        subprocess.run([openvaf(), f"{model}.va", "--output", str(osdi_dir / f"{model}.osdi")],
                       cwd=MODELS.parent / "verilog-a" / src, check=True, capture_output=True)
sorted(p.name for p in osdi_dir.glob("*.osdi"))


['mosvar.osdi', 'psp103.osdi', 'psp103_nqs.osdi', 'r3_cmc.osdi']

`notebooks/spice.py` is a ~80-line helper shared by all notebooks:
- `spice.header()` returns the model library line.
- `spice.run(netlist)` runs `ngspice -b` in a temp dir, with a `.spiceinit` that loads the OSDI models.
- `spice.read(file)` loads `wrdata` output into a DataFrame.
- `spice.mos(inst, model, *params)` names the PSP103 operating-point vectors of a transistor, e.g. `@n.xm1.nsg13_lv_nmos[gm]`. `spice.read` shortens them to `xm1.gm`.

A quick operating-point smoke test on a minimum-length device:


In [6]:
os.environ["PDK_ROOT"], os.environ["PDK"] = str(PDK_ROOT), PDK_VARIANT
import spice

wd = spice.run(spice.header() + f'''
* smoke test: one minimum-length sg13_lv_nmos biased in saturation
VG g 0 0.6
VD d 0 0.6
XM1 d g 0 0 {spice.NMOS} w=1u l=0.13u ng=1
.control
op
print {spice.mos("xm1", spice.NMOS, "ids", "gm", "gds", "vth", "vdss")}
.endc
.end
''')
print("\n".join(l for l in (wd / "tb.log").read_text().splitlines() if l.startswith("@n.")))


@n.xm1.nsg13_lv_nmos[ids] = 1.836885e-05
@n.xm1.nsg13_lv_nmos[gm] = 1.904314e-04
@n.xm1.nsg13_lv_nmos[gds] = 1.083106e-05
@n.xm1.nsg13_lv_nmos[vth] = 4.562108e-01
@n.xm1.nsg13_lv_nmos[vdss] = 1.841704e-01


## 3. LUT extraction testbench

novaad sizes devices from tables of small-signal parameters over a grid of `vgs × vds × vsb × lch`, all characterized at one reference width `REF_WIDTH`. The columns it needs are:

| column | meaning | source |
|---|---|---|
| `vgs vds vsb lch` | grid axes | sweep |
| `ids gm gds` | drain current and conductances | PSP103 op-point |
| `cgg cgs cgd` | gate capacitances (intrinsic + overlap) | `cgg+cgsol+cgdol`, `cgs+cgsol`, `cgd+cgdol` |
| `cdb csb` | junction capacitances | `cjd`, `cjs` |
| `gmoverid jd ft av ron fom_bw` | derived figures | `gm/ids`, `ids/W`, `gm/(2π·cgg)`, `gm/gds`, `1/gds`, `ft·av` |

PSP103 reports the intrinsic capacitances as positive values and the gate overlap capacitances `cgsol`/`cgdol` separately, so the overlaps are added back.

PMOS is swept with negated voltages. novaad takes `abs()` of the whole LUT, so both types end up with positive values.


In [7]:
# Global LUT sweep settings: edit these
VDD = 1.2                                   # sweep range for |vgs| and |vds| (1.2 V core devices)
REF_WIDTH = 10e-6                           # characterization width; novaad scales W from it
LCH = [0.13e-6, 0.18e-6, 0.25e-6, 0.5e-6, 1e-6, 2e-6]   # channel lengths (0.13 um is the minimum)
VSB = [0.0, 0.2, 0.4, 0.6]                  # source-bulk voltages
VGS_STEP = 0.025
VDS_STEP = 0.05
CORNER = "mos_tt"
TEMP = 27
OUT_DIR = Path("luts")


In [8]:
DEVICES = {"nch": spice.NMOS, "pch": spice.PMOS}
PARAMS = ["ids", "gm", "gds", "cgg", "cgs", "cgd", "cgsol", "cgdol", "cjd", "cjs"]


def testbench(model: str, lch: float, vsb: float, sign: int) -> str:
    # sign=+1 for nmos, -1 for pmos (source grounded, all voltages negated)
    vecs = " ".join(["v(g)", "v(d)", spice.mos("xm1", model, *PARAMS)])
    return spice.header(CORNER, TEMP) + f'''
* {model} LUT sweep: L={lch}, VSB={vsb}
VG g 0 0
VD d 0 0
VB b 0 {-sign * vsb}
XM1 d g 0 b {model} w={REF_WIDTH} l={lch} ng=1
.control
set wr_singlescale
set wr_vecnames
save {vecs}
dc VG 0 {sign * VDD} {sign * VGS_STEP} VD {sign * VDS_STEP} {sign * VDD} {sign * VDS_STEP}
wrdata lut.txt {vecs}
.endc
.end
'''


print(testbench(spice.NMOS, LCH[0], VSB[0], +1))

* novaad testbench
.lib /Users/gfdasdias/.ciel/ihp-sg13g2/libs.tech/ngspice/models/cornerMOSlv.lib mos_tt
.temp 27
.option savecurrents

* sg13_lv_nmos LUT sweep: L=1.3e-07, VSB=0.0
VG g 0 0
VD d 0 0
VB b 0 -0.0
XM1 d g 0 b sg13_lv_nmos w=1e-05 l=1.3e-07 ng=1
.control
set wr_singlescale
set wr_vecnames
save v(g) v(d) @n.xm1.nsg13_lv_nmos[ids] @n.xm1.nsg13_lv_nmos[gm] @n.xm1.nsg13_lv_nmos[gds] @n.xm1.nsg13_lv_nmos[cgg] @n.xm1.nsg13_lv_nmos[cgs] @n.xm1.nsg13_lv_nmos[cgd] @n.xm1.nsg13_lv_nmos[cgsol] @n.xm1.nsg13_lv_nmos[cgdol] @n.xm1.nsg13_lv_nmos[cjd] @n.xm1.nsg13_lv_nmos[cjs]
dc VG 0 1.2 0.025 VD 0.05 1.2 0.05
wrdata lut.txt v(g) v(d) @n.xm1.nsg13_lv_nmos[ids] @n.xm1.nsg13_lv_nmos[gm] @n.xm1.nsg13_lv_nmos[gds] @n.xm1.nsg13_lv_nmos[cgg] @n.xm1.nsg13_lv_nmos[cgs] @n.xm1.nsg13_lv_nmos[cgd] @n.xm1.nsg13_lv_nmos[cgsol] @n.xm1.nsg13_lv_nmos[cgdol] @n.xm1.nsg13_lv_nmos[cjd] @n.xm1.nsg13_lv_nmos[cjs]
.endc
.end



The sweep starts at `|vds| = VDS_STEP` because at `vds = 0` the drain current is zero and `gm/id` would be infinite. Each `(lch, vsb)` pair is one ngspice run, and each run does a nested `vgs × vds` DC sweep.

In [9]:
from math import pi
from pandas import DataFrame, concat


def extract(dev: str) -> DataFrame:
    frames = []
    for lch in LCH:
        for vsb in VSB:
            wd = spice.run(testbench(DEVICES[dev], lch, vsb, +1 if dev == "nch" else -1))
            raw = spice.read(wd / "lut.txt")
            frames.append(DataFrame({
                "vgs": raw["v(g)"], "vds": raw["v(d)"], "vsb": vsb, "lch": lch,
                "ids": raw["xm1.ids"], "gm": raw["xm1.gm"], "gds": raw["xm1.gds"],
                "cgg": raw["xm1.cgg"] + raw["xm1.cgsol"] + raw["xm1.cgdol"],
                "cgs": raw["xm1.cgs"] + raw["xm1.cgsol"],
                "cgd": raw["xm1.cgd"] + raw["xm1.cgdol"],
                "cdb": raw["xm1.cjd"], "csb": raw["xm1.cjs"],
            }))
    lut = concat(frames, ignore_index=True).abs()
    lut[["vgs", "vds"]] = lut[["vgs", "vds"]].round(6)   # exact grid values (the GUI needs them)
    lut = lut[lut["ids"] > 0].copy()
    lut["gmoverid"] = lut["gm"] / lut["ids"]
    lut["jd"] = lut["ids"] / REF_WIDTH
    lut["ft"] = lut["gm"] / (2 * pi * lut["cgg"])
    lut["av"] = lut["gm"] / lut["gds"]
    lut["ron"] = 1 / lut["gds"]
    lut["fom_bw"] = lut["ft"] * lut["av"]
    return lut


OUT_DIR.mkdir(exist_ok=True)
LUTS = {}
for dev in DEVICES:
    LUTS[dev] = extract(dev)
    LUTS[dev].to_csv(OUT_DIR / f"sg13_lv_{dev}.csv", index=False)
    print(f"{dev}: {len(LUTS[dev])} rows -> {OUT_DIR / f'sg13_lv_{dev}.csv'}")
LUTS["nch"].describe().T[["min", "max"]]

nch: 28224 rows -> luts/sg13_lv_nch.csv


pch: 28224 rows -> luts/sg13_lv_pch.csv


,min,max
vgs,0.000000e+00,1.200000e+00
vds,5.000000e-02,1.200000e+00
vsb,0.000000e+00,6.000000e-01
lch,1.300000e-07,2.000000e-06
ids,1.386322e-11,4.791522e-03
gm,3.924424e-10,9.509557e-03
gds,6.843425e-11,1.533479e-02
cgg,1.194297e-14,2.554982e-13
cgs,6.048402e-15,1.750709e-13
cgd,5.253428e-15,1.245503e-13


To use the LUTs from the novaad **CLI**, add them to your `pyproject.toml`:

In [10]:
print("[tool.novaad.nch]")
print(f'lut_path = "{(OUT_DIR / "sg13_lv_nch.csv").resolve()}"')
print(f"ref_width = {REF_WIDTH}\n")
print("[tool.novaad.pch]")
print(f'lut_path = "{(OUT_DIR / "sg13_lv_pch.csv").resolve()}"')
print(f"ref_width = {REF_WIDTH}")

[tool.novaad.nch]
lut_path = "/Users/gfdasdias/Documents/GF_Software/novaad/notebooks/luts/sg13_lv_nch.csv"
ref_width = 1e-05

[tool.novaad.pch]
lut_path = "/Users/gfdasdias/Documents/GF_Software/novaad/notebooks/luts/sg13_lv_pch.csv"
ref_width = 1e-05


## Sanity check: load the LUTs with novaad

These are the classic gm/id design charts at `|vds| = VDD/2` and `vsb = 0`, one curve per channel length. NMOS gm/id should approach roughly 25–30 V⁻¹ in weak inversion, and minimum-length `ft` should reach tens of GHz.

In [11]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.palettes import Category10
from bokeh.plotting import figure
from novaad import Device

output_notebook(hide_banner=True)

CHARTS = [  # (x, y, x_axis_type, y_axis_type)
    ("vgs", "gmoverid", "linear", "linear"),
    ("gmoverid", "jd", "linear", "log"),
    ("gmoverid", "ft", "linear", "log"),
    ("gmoverid", "av", "linear", "log"),
]

rows = []
for dev in DEVICES:
    device = Device(OUT_DIR / f"sg13_lv_{dev}.csv", device_type=dev, ref_width=REF_WIDTH)
    lut = device.lut
    vds = lut["vds"].unique()[abs(lut["vds"].unique() - VDD / 2).argmin()]
    row = []
    for x, y, xt, yt in CHARTS:
        p = figure(title=f"{dev}: {y} vs {x}", x_axis_label=x, y_axis_label=y,
                   x_axis_type=xt, y_axis_type=yt, width=330, height=260)
        for lch, color in zip(LCH, Category10[10]):
            d = lut.query("abs(vds - @vds) < 1e-6 and vsb == 0 and abs(lch - @lch) < 1e-9").sort_values("vgs")
            p.line(d[x], d[y], color=color, legend_label=f"{lch * 1e6:g} um")
        p.legend.label_text_font_size = "7pt"
        p.legend.location = "top_left" if y != "gmoverid" else "top_right"
        row.append(p)
    rows.append(row)
show(gridplot(rows))